# 02 SQL in R Analytics
This notebook demonstrates SQL queries inside R using sqldf. It loads the same NorthStar CSV data and produces query outputs required for the SQL in R marking section.

In [ ]:
install.packages(c("sqldf", "dplyr", "ggplot2", "readr"), repos="https://cloud.r-project.org")

In [ ]:
library(sqldf)
library(dplyr)
library(ggplot2)
library(readr)
DATA_DIR <- "/content/northstar_dataset"
if (!dir.exists(DATA_DIR)) {
  dir.create(DATA_DIR)
  cat("Upload CSV files into /content/northstar_dataset using the Colab file panel. Then rerun this cell.
")
}
orders <- read_csv(file.path(DATA_DIR, "orders.csv"), show_col_types = FALSE)
deliveries <- read_csv(file.path(DATA_DIR, "deliveries.csv"), show_col_types = FALSE)
complaints <- read_csv(file.path(DATA_DIR, "complaints.csv"), show_col_types = FALSE)
hubs <- read_csv(file.path(DATA_DIR, "hubs.csv"), show_col_types = FALSE)
print(dim(orders)); print(dim(deliveries)); print(dim(complaints)); print(dim(hubs))

In [ ]:
# Query 1: delivery performance by hub
hub_sql <- sqldf("
SELECT h.hub_id, h.hub_name, h.zone,
       COUNT(d.delivery_id) AS total_deliveries,
       SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed') THEN 1 ELSE 0 END) * 1.0 / COUNT(d.delivery_id) AS problem_rate,
       AVG(d.manual_route_override_count) AS avg_route_overrides,
       AVG(d.customer_rating_post_delivery) AS avg_rating,
       AVG(d.fuel_or_charge_cost) AS avg_cost
FROM deliveries d
JOIN hubs h ON d.hub_id = h.hub_id
GROUP BY h.hub_id, h.hub_name, h.zone
ORDER BY problem_rate DESC
")
hub_sql

In [ ]:
# Query 2: service type risk profile
service_sql <- sqldf("
SELECT o.service_type, COUNT(*) AS total_orders,
       SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed') THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS problem_rate,
       AVG(o.order_value) AS avg_order_value,
       AVG(d.fuel_or_charge_cost) AS avg_cost,
       AVG(d.customer_rating_post_delivery) AS avg_rating
FROM orders o
JOIN deliveries d ON o.order_id = d.order_id
GROUP BY o.service_type
ORDER BY problem_rate DESC
")
service_sql

In [ ]:
# Query 3: complaint pressure by service type
complaint_sql <- sqldf("
SELECT o.service_type, c.complaint_type, COUNT(c.complaint_id) AS complaint_count,
       AVG(c.resolution_days) AS avg_resolution_days,
       SUM(c.compensation_amount) AS total_compensation
FROM complaints c
JOIN orders o ON c.order_id = o.order_id
GROUP BY o.service_type, c.complaint_type
ORDER BY complaint_count DESC
LIMIT 15
")
complaint_sql

In [ ]:
ggplot(hub_sql, aes(x=reorder(hub_name, -problem_rate), y=problem_rate)) +
  geom_col() +
  labs(title="Problem delivery rate by hub", x="Hub", y="Problem rate") +
  theme(axis.text.x = element_text(angle=35, hjust=1))

In [ ]:
ggplot(service_sql, aes(x=reorder(service_type, -problem_rate), y=problem_rate)) +
  geom_col() +
  labs(title="Problem delivery rate by service type", x="Service type", y="Problem rate")